# Spin-Echo Simulation — Reproduction

**BlochSimulator Version**: 1.1.0

**Mode**: Re-run simulation from parameters

This notebook reproduces the simulation from scratch using the exported parameters.

## Installation

If you haven't installed the `blochsimulator` package yet, you can do so using pip:

```bash
# From GitHub (latest version)
!pip install git+https://github.com/LucaNagel/bloch_sim_gui.git

# From local directory (if you have the source code)
# !pip install .
```

## Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
from pathlib import Path
from blochsimulator import (
    BlochSimulator, TissueParameters,
    SpinEcho, SpinEchoTipAxis, GradientEcho,
    SliceSelectRephase, design_rf_pulse
)

# Set matplotlib style
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## Simulation Parameters

In [ ]:
# Define simulation parameters

# Tissue parameters
tissue_name = 'White Matter (3T)'
t1 = 0.830000  # seconds
t2 = 0.070000  # seconds
density = 1.000

# Sequence parameters
sequence_type = 'Spin Echo'
te = 0.020000  # seconds
tr = 0.100000  # seconds
flip_angle = 90.0  # degrees

# Simulation parameters
num_positions = 2
num_frequencies = 7
time_step_us = 200.000
mode = 2 if 'time-resolved' == 'time-resolved' else 0

# Parameter dictionary (used for some sequence types)
sequence_params = {
    'sequence_type': 'Spin Echo',
    'te': 0.02,
    'tr': 0.1,
    'flip_angle': 90.0,
}


## Initialize Simulator

In [ ]:
# Create simulator
use_parallel = False
num_threads = 1

sim = BlochSimulator(use_parallel=use_parallel, num_threads=num_threads)

# Create tissue
tissue = TissueParameters(
    name=tissue_name,
    t1=t1,
    t2=t2,
    density=density
)

print(f"Simulator initialized")
print(f"  Tissue: {tissue.name}")
print(f"  T1: {tissue.t1*1000:.1f} ms, T2: {tissue.t2*1000:.1f} ms")


## Define Pulse Sequence

In [ ]:
# Create Spin Echo sequence
sequence = SpinEcho(
    te=te,
    tr=tr
)
print(f"Spin Echo sequence: TE={te*1000:.1f} ms, TR={tr*1000:.1f} ms")


## Spatial and Frequency Sampling

In [ ]:
# Define spatial positions
positions = np.zeros((num_positions, 3))
if num_positions > 1:
    positions[:, 2] = np.linspace(-0.010000, 0.010000, num_positions)

# Define off-resonance frequencies
if num_frequencies > 1:
    frequencies = np.linspace(-50.0, 50.0, num_frequencies)
else:
    frequencies = np.array([0.0])

print(f"Sampling:")
print(f"  Positions: {num_positions}")
print(f"  Frequencies: {num_frequencies}")


## Run Simulation

In [ ]:
# Run simulation
print("\nRunning simulation...")

result = sim.simulate(
    sequence,
    tissue,
    positions=positions,
    frequencies=frequencies,
    mode=mode,
    dt=time_step_us * 1e-6
)

# Extract results for easier access
from dataclasses import asdict
data = {
    'mx': result['mx'],
    'my': result['my'],
    'mz': result['mz'],
    'signal': result['signal'],
    'time': result['time'],
    'positions': result['positions'],
    'frequencies': result['frequencies'],
    'tissue': asdict(tissue),
    'sequence_params': sequence_params,
    'simulation_params': {
        'num_positions': len(positions),
        'num_frequencies': len(frequencies),
        'mode': 'time-resolved' if mode == 2 else 'endpoint',
        'use_parallel': use_parallel,
        'num_threads': num_threads
    }
}

print(f"Simulation complete!")
print(f"  Result shape: {result['mx'].shape}")
print(f"  Duration: {result['time'][-1]*1000:.3f} ms")


## Xarray Dataset

In [ ]:
# Convert to xarray Dataset for advanced analysis
# Extract info from metadata
n_pos = data.get('simulation_params', {}).get('num_positions', 1)
n_freq = data.get('simulation_params', {}).get('num_frequencies', 1)
time = data.get('time')
n_time = len(time) if time is not None else 0

# Create DataArray for each component
vars = {}
coords = {}
if time is not None: coords['time'] = time

for k in ['mx', 'my', 'mz', 'signal']:
    v = data[k]
    dims = []

    # Try to intelligently name dimensions
    for i, dim_len in enumerate(v.shape):
        if n_time > 0 and dim_len == n_time:
            dims.append('time')
        elif n_pos > 1 and dim_len == n_pos:
            dims.append('position')
        elif n_freq > 1 and dim_len == n_freq:
            dims.append('frequency')
        else:
            dims.append(f'dim_{i}')

    vars[k] = (dims, v)

ds = xr.Dataset(vars, coords=coords)
# Add metadata
ds.attrs.update(data.get('simulation_params', {}))
ds.attrs.update(data.get('sequence_params', {}))

print('Xarray Dataset created as "ds":')
print(ds)

## Visualization

In [ ]:
# Plot magnetization evolution
# Always choose central index for position and frequency
position_idx = data['positions'].shape[0] // 2
freq_idx = len(data['frequencies']) // 2

# Get actual values for title
pos_z_cm = data['positions'][position_idx, 2] * 100
freq_hz = data['frequencies'][freq_idx]

if data['mx'].ndim == 3:  # Time-resolved
    time_ms = data['time'] * 1000
    mx = data['mx'][:, position_idx, freq_idx]
    my = data['my'][:, position_idx, freq_idx]
    mz = data['mz'][:, position_idx, freq_idx]
    mxy = np.sqrt(mx**2 + my**2)

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    axes[0, 0].plot(time_ms, mx, 'b-', linewidth=1.5)
    axes[0, 0].set_xlabel('Time (ms)')
    axes[0, 0].set_ylabel('Mx')
    axes[0, 0].set_title('Transverse Magnetization (x)')
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(time_ms, my, 'r-', linewidth=1.5)
    axes[0, 1].set_xlabel('Time (ms)')
    axes[0, 1].set_ylabel('My')
    axes[0, 1].set_title('Transverse Magnetization (y)')
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].plot(time_ms, mz, 'g-', linewidth=1.5)
    axes[1, 0].set_xlabel('Time (ms)')
    axes[1, 0].set_ylabel('Mz')
    axes[1, 0].set_title('Longitudinal Magnetization')
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].plot(time_ms, mxy, color='purple', linewidth=1.5)
    axes[1, 1].set_xlabel('Time (ms)')
    axes[1, 1].set_ylabel('|Mxy|')
    axes[1, 1].set_title('Transverse Magnitude')
    axes[1, 1].grid(True, alpha=0.3)

    plt.suptitle(f'Magnetization Evolution - Pos: {pos_z_cm:.2f} cm, Freq: {freq_hz:.1f} Hz',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("Endpoint data - no time evolution to plot")


## Signal Analysis

In [ ]:
# Plot signal
# Re-use central indices
position_idx = data['positions'].shape[0] // 2
freq_idx = len(data['frequencies']) // 2

pos_z_cm = data['positions'][position_idx, 2] * 100
freq_hz = data['frequencies'][freq_idx]

if data['signal'].ndim == 3:  # Time-resolved
    signal = data['signal'][:, position_idx, freq_idx]
    time_ms = data['time'] * 1000

    fig, axes = plt.subplots(2, 1, figsize=(12, 8))

    axes[0].plot(time_ms, np.real(signal), 'b-', label='Real', linewidth=1.5)
    axes[0].plot(time_ms, np.imag(signal), 'r-', label='Imaginary', linewidth=1.5)
    axes[0].set_xlabel('Time (ms)')
    axes[0].set_ylabel('Signal')
    axes[0].set_title('Complex Signal Components')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(time_ms, np.abs(signal), color='purple', linewidth=1.5)
    axes[1].set_xlabel('Time (ms)')
    axes[1].set_ylabel('|Signal|')
    axes[1].set_title('Signal Magnitude')
    axes[1].grid(True, alpha=0.3)

    plt.suptitle(f'MRI Signal - Pos: {pos_z_cm:.2f} cm, Freq: {freq_hz:.1f} Hz',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("Endpoint data - no time evolution to plot")


## Save Results (Optional)

In [ ]:
# Uncomment to save results
# sim.save_results('simulation_results.h5', sequence_params, simulation_params)
# print('Results saved!')